# 10N-AI: Asistente Inteligente de Machine Learning
### Proyecto I - Inteligencia Artificial | ITCR San Carlos
### Modelo 01: Predicción del Precio de Venta de Automóviles Usados
**Estudiantes:** Carlos Arce & Joshua Picado  
**Profesor:** Ing. Efrén Jiménez Delgado

# Análisis del Problema

### Formulación del Problema
La tasación y fijación de precios en el mercado de vehículos usados es una tarea crítica caracterizada por la asimetría de información y la depreciación acelerada de los activos. Establecer precios incorrectos genera pérdidas económicas por estancamiento de inventario o subvaloración del bien. El objetivo de este modelo es predecir con alta precisión el **precio de venta (`Selling_Price`)** en miles/lakhs a partir de variables mecánicas y comerciales como el precio de agencia (`Present_Price`), kilometraje acumulado (`Kms_Driven`), antigüedad, tipo de combustible, tipo de transmisión y cantidad de propietarios anteriores.

Este modelo funcionará como una habilidad central del asistente inteligente **10N-AI**, permitiendo a los usuarios consultar por voz o texto: *"10N-AI, cotizá el precio estimado para un auto 2017 con 45,000 km a gasolina"*.

### Selección del Modelo Visto en Clase
Para este problema de **aprendizaje supervisado de regresión**, se selecciona **Random Forest Regressor (Bosque Aleatorio)** como modelo principal, y **Regresión Lineal con Regularización Ridge** como modelo base comparativo.

#### Justificación de la Selección:
1. **Modelado de No Linealidades Complejas:** La depreciación vehicular no sigue una curva lineal simple; presenta variaciones pronunciadas según umbrales de kilometraje y tipo de combustible. Los ensambles basados en árboles capturan estas particiones no lineales de forma natural.
2. **Robustez ante Valores Extremos:** Vehículos de gama alta con precios muy distantes de la media sesgan fuertemente los modelos de regresión paramétrica ordinaria; Random Forest mitiga el impacto de estos outliers mediante el promedio de múltiples estimadores independientes.
3. **Interpretabilidad Cuantificable:** Permite extraer la importancia relativa de variables (Feature Importance), insumo indispensable para que el asistente 10N-AI justifique al usuario qué factores elevaron o castigaron el valor sugerido.

In [1]:
# @title Importación de Librerías y Configuración Global { display-mode: "form" }
import os
import sys
import warnings
import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

warnings.filterwarnings('ignore')
print("Librerías importadas exitosamente para 10N-AI.")

Librerías importadas exitosamente para 10N-AI.


In [2]:
# @title Conexión a Google Drive y Carga del Dataset { display-mode: "form" }
from google.colab import drive
drive.mount('/content/drive')

ruta_dataset = '/content/drive/MyDrive/Datasets/car data.csv'

if not os.path.exists(ruta_dataset):
    raise FileNotFoundError(
        f"No se encontró el archivo en: {ruta_dataset}. "
        "Asegúrate de haber montado Google Drive y de tener el archivo en la carpeta 'Datasets'."
    )

print(f"Cargando dataset desde Google Drive: {ruta_dataset}")
df = pd.read_csv(ruta_dataset)
print(f"Dataset cargado con éxito. Dimensiones: {df.shape[0]} filas y {df.shape[1]} columnas.")
display(df.head())

Mounted at /content/drive
Cargando dataset desde Google Drive: /content/drive/MyDrive/Datasets/car data.csv
Dataset cargado con éxito. Dimensiones: 301 filas y 9 columnas.


,Car_Name,Year,Selling_Price,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner
0,ritz,2014,3.35,5.59,27000,Petrol,Dealer,Manual,0
1,sx4,2013,4.75,9.54,43000,Diesel,Dealer,Manual,0
2,ciaz,2017,7.25,9.85,6900,Petrol,Dealer,Manual,0
3,wagon r,2011,2.85,4.15,5200,Petrol,Dealer,Manual,0
4,swift,2014,4.60,6.87,42450,Diesel,Dealer,Manual,0


# Entendimiento de los Datos

### Diccionario de Variables
* **`Car_Name` (Nominal):** Nombre o marca del modelo del automóvil (e.g., ritz, sx4, ciaz).
* **`Year` (Numérica discreta):** Año de fabricación del automóvil.
* **`Selling_Price` (Numérica continua - Target):** Precio de venta al que se comercializó el vehículo.
* **`Present_Price` (Numérica continua):** Precio actual de lista del vehículo nuevo en agencia/showroom.
* **`Kms_Driven` (Numérica continua):** Kilometraje acumulado recorrido por el vehículo.
* **`Fuel_Type` (Categórica nominal):** Tipo de combustible (`Petrol`, `Diesel`, `CNG`).
* **`Seller_Type` (Categórica nominal):** Canal de venta (`Dealer` = Concesionario, `Individual` = Particular).
* **`Transmission` (Categórica nominal):** Tipo de transmisión (`Manual`, `Automatic`).
* **`Owner` (Numérica discreta):** Cantidad de propietarios anteriores registrados (0, 1, 3).

In [3]:
# @title Inspección de Estructura, Tipos de Datos y Valores Nulos { display-mode: "form" }
print("=== INFORMACIÓN GENERAL DEL DATASET ===")
print(df.info())
print("\n" + "="*60 + "\n")

print("=== CONTEO DE VALORES NULOS POR COLUMNA ===")
nulos = df.isnull().sum()
print(nulos[nulos > 0] if nulos.sum() > 0 else "No existen valores nulos en el dataset.")
print("\n" + "="*60 + "\n")

print("=== REGISTROS DUPLICADOS ===")
duplicados = df.duplicated().sum()
print(f"Cantidad de filas duplicadas: {duplicados}")
if duplicados > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Duplicados eliminados. Nuevas dimensiones: {df.shape}")
print("\n" + "="*60 + "\n")

print("=== RESUMEN ESTADÍSTICO DE VARIABLES NUMÉRICAS ===")
display(df.describe().T.round(2))

=== INFORMACIÓN GENERAL DEL DATASET ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 301 entries, 0 to 300
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Car_Name       301 non-null    object 
 1   Year           301 non-null    int64  
 2   Selling_Price  301 non-null    float64
 3   Present_Price  301 non-null    float64
 4   Kms_Driven     301 non-null    int64  
 5   Fuel_Type      301 non-null    object 
 6   Seller_Type    301 non-null    object 
 7   Transmission   301 non-null    object 
 8   Owner          301 non-null    int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 21.3+ KB
None


=== CONTEO DE VALORES NULOS POR COLUMNA ===
No existen valores nulos en el dataset.


=== REGISTROS DUPLICADOS ===
Cantidad de filas duplicadas: 2
Duplicados eliminados. Nuevas dimensiones: (299, 9)


=== RESUMEN ESTADÍSTICO DE VARIABLES NUMÉRICAS ===


,count,mean,std,min,25%,50%,75%,max
Year,299.0,2013.62,2.90,2003.00,2012.00,2014.00,2016.00,2018.0
Selling_Price,299.0,4.59,4.98,0.10,0.85,3.51,6.00,35.0
Present_Price,299.0,7.54,8.57,0.32,1.20,6.10,9.84,92.6
Kms_Driven,299.0,36916.75,39015.17,500.00,15000.00,32000.00,48883.50,500000.0
Owner,299.0,0.04,0.25,0.00,0.00,0.00,0.00,3.0


### Hallazgos Principales del Entendimiento de Datos
1. **Completitud:** El conjunto de datos no presenta valores faltantes en ninguna de sus columnas.
2. **Dispersión de Precios:** El precio de venta oscila entre 0.1 y 35.0 lakhs, con una media de 4.66 y una mediana inferior, evidenciando asimetría positiva por vehículos de gama premium.
3. **Uso y Antigüedad:** El kilometraje promedio se sitúa en ~36,900 km, con vehículos que alcanzan hasta 500,000 km, representando una amplia gama de desgaste físico.

# Exploración de los Datos

In [5]:
# @title Distribución Interactiva del Precio de Venta (Variable Objetivo) { display-mode: "form" }
fig_target = px.histogram(
    df,
    x='Selling_Price',
    nbins=35,
    marginal='box',
    title='<b>Distribución del Precio de Venta (Selling_Price) con Diagrama Marginal</b>',
    labels={'Selling_Price': 'Precio de Venta'},
    color_discrete_sequence=['#2563eb']
)

fig_target.update_layout(
    template='plotly_white',
    xaxis_title='Precio de Venta (Lakhs)',
    yaxis_title='Frecuencia',
    showlegend=False
)
fig_target.show()

# Análisis
print("--- ANÁLISIS DE LA VARIABLE OBJETIVO (SELLING_PRICE) ---")
print("• Concentración: La gran mayoría de transacciones se concentran en el rango de 0 a 8 lakhs.")
print("• Asimetría Positiva: Existen vehículos con precios superiores a 15 lakhs que actúan como outliers superiores.")
print("• Mediana vs Media: La mediana de venta ronda los 3.51 lakhs, demostrando que los precios altos inflan el promedio aritmético.")

--- ANÁLISIS DE LA VARIABLE OBJETIVO (SELLING_PRICE) ---
• Concentración: La gran mayoría de transacciones se concentran en el rango de 0 a 8 lakhs.
• Asimetría Positiva: Existen vehículos con precios superiores a 15 lakhs que actúan como outliers superiores.
• Mediana vs Media: La mediana de venta ronda los 3.51 lakhs, demostrando que los precios altos inflan el promedio aritmético.


In [6]:
# @title Relación Interactiva entre Precio de Venta y Precio en Agencia { display-mode: "form" }
fig_scatter_present = px.scatter(
    df,
    x='Present_Price',
    y='Selling_Price',
    color='Fuel_Type',
    size='Selling_Price',
    hover_data=['Car_Name', 'Year', 'Kms_Driven', 'Transmission'],
    title='<b>Precio de Venta vs. Precio de Agencia según Tipo de Combustible</b>',
    labels={'Present_Price': 'Precio en Agencia (Nuevo)', 'Selling_Price': 'Precio de Venta (Usado)'},
    color_discrete_map={'Petrol': '#3b82f6', 'Diesel': '#10b981', 'CNG': '#f59e0b'}
)

fig_scatter_present.update_layout(
    template='plotly_white',
    xaxis_title='Precio de Agencia (Lakhs)',
    yaxis_title='Precio de Venta (Lakhs)'
)
fig_scatter_present.show()

# Análisis
print("--- ANÁLISIS: PRECIO DE AGENCIA Y TIPO DE COMBUSTIBLE ---")
print("• Alta Correlación Positiva: A mayor valor de nuevo en agencia, mayor precio de reventa.")
print("• Dominio Diésel en Precios Altos: Los autos a Diésel se concentran en segmentos de mayor valor y mantienen mejor su precio.")
print("• Estabilidad Gasolina (Petrol): Representa el volumen de mercado más grande, con precios de venta consistentes menores a 10 lakhs.")

--- ANÁLISIS: PRECIO DE AGENCIA Y TIPO DE COMBUSTIBLE ---
• Alta Correlación Positiva: A mayor valor de nuevo en agencia, mayor precio de reventa.
• Dominio Diésel en Precios Altos: Los autos a Diésel se concentran en segmentos de mayor valor y mantienen mejor su precio.
• Estabilidad Gasolina (Petrol): Representa el volumen de mercado más grande, con precios de venta consistentes menores a 10 lakhs.


In [7]:
# @title Distribución Interactiva de Precios según Transmisión y Vendedor { display-mode: "form" }
fig_box_trans = px.box(
    df,
    x='Transmission',
    y='Selling_Price',
    color='Seller_Type',
    title='<b>Comparativa de Precios según Tipo de Transmisión y Canal de Venta</b>',
    labels={'Transmission': 'Transmisión', 'Selling_Price': 'Precio de Venta', 'Seller_Type': 'Vendedor'},
    color_discrete_map={'Dealer': '#6366f1', 'Individual': '#ec4899'}
)

fig_box_trans.update_layout(
    template='plotly_white',
    xaxis_title='Tipo de Transmisión',
    yaxis_title='Precio de Venta (Lakhs)'
)
fig_box_trans.show()

# Análisis
print("--- ANÁLISIS: TRANSMISIÓN Y CANAL DE VENTA ---")
print("• Transmisión Automática: Presenta medianas de precio significativamente superiores a la manual.")
print("• Canal Concesionario (Dealer): Logra precios de venta más altos en comparación con particulares (Individual),")
print("  quienes comercializan vehículos de menor valor o con mayor urgencia de liquidez.")

--- ANÁLISIS: TRANSMISIÓN Y CANAL DE VENTA ---
• Transmisión Automática: Presenta medianas de precio significativamente superiores a la manual.
• Canal Concesionario (Dealer): Logra precios de venta más altos en comparación con particulares (Individual),
  quienes comercializan vehículos de menor valor o con mayor urgencia de liquidez.


In [10]:
# @title Matriz de Correlación Interactiva { display-mode: "form" }
df_corr = df.copy()
df_corr['Vehicle_Age'] = 2026 - df_corr['Year']
cols_num = ['Selling_Price', 'Present_Price', 'Kms_Driven', 'Vehicle_Age', 'Owner']
matriz_corr = df_corr[cols_num].corr()

fig_heatmap = px.imshow(
    matriz_corr,
    text_auto='.2f',
    color_continuous_scale='Blues',
    zmin=-1.0,
    zmax=1.0,
    title='<b>Matriz de Correlación Interactiva de Características Numéricas</b>'
)

fig_heatmap.update_layout(
    template='plotly_white',
    width=750,
    height=600
)
fig_heatmap.show()

# Análisis resumido integrado en consola
print("--- ANÁLISIS DE LA MATRIZ DE CORRELACIÓN ---")
print("• Present_Price (r = +0.88): Relación positiva determinante con Selling_Price, es el predictor más fuerte.")
print("• Vehicle_Age (r = -0.23): Correlación negativa clara; a mayor antigüedad del vehículo, menor valor en el mercado.")
print("• Kms_Driven (r = +0.03 global): Afecta la depreciación individualmente, pero en escala multivariable interactúa con la edad.")

--- ANÁLISIS DE LA MATRIZ DE CORRELACIÓN ---
• Present_Price (r = +0.88): Relación positiva determinante con Selling_Price, es el predictor más fuerte.
• Vehicle_Age (r = -0.23): Correlación negativa clara; a mayor antigüedad del vehículo, menor valor en el mercado.
• Kms_Driven (r = +0.03 global): Afecta la depreciación individualmente, pero en escala multivariable interactúa con la edad.


# Modelo de machine learning

### Ingeniería de Características y Preparación
1. **Transformación Temporal (`Vehicle_Age`):** Se calcula `2026 - Year` para obtener la edad exacta en años.
2. **Remoción de Identificadores:** Se descarta `Car_Name` para evitar dispersión y sobreajuste por nombres de modelos con pocas muestras.
3. **Codificación de Categóricas:** Aplicación de One-Hot Encoding (`drop_first=True`) sobre `Fuel_Type`, `Seller_Type` y `Transmission`.

In [11]:
# @title Partición de Datos Train / Test y Preparación { display-mode: "form" }
df_ml = df.copy()
df_ml['Vehicle_Age'] = 2026 - df_ml['Year']
df_ml.drop(['Car_Name', 'Year'], axis=1, inplace=True)
df_ml = pd.get_dummies(df_ml, drop_first=True)

X = df_ml.drop('Selling_Price', axis=1)
y = df_ml['Selling_Price']

columnas_predictoras = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"Dimensiones de entrenamiento (X_train): {X_train.shape}")
print(f"Dimensiones de prueba (X_test): {X_test.shape}")
print(f"Columnas predictoras ({len(columnas_predictoras)}):", columnas_predictoras)

Dimensiones de entrenamiento (X_train): (239, 8)
Dimensiones de prueba (X_test): (60, 8)
Columnas predictoras (8): ['Present_Price', 'Kms_Driven', 'Owner', 'Vehicle_Age', 'Fuel_Type_Diesel', 'Fuel_Type_Petrol', 'Seller_Type_Individual', 'Transmission_Manual']


In [12]:
# @title Búsqueda de Hiperparámetros Óptimos (GridSearchCV) { display-mode: "form" }
espacio_hiperparametros = {
    'n_estimators': [50, 100, 150],
    'max_depth': [None, 6, 10, 14],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

grid_rf = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=espacio_hiperparametros,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)

print("=== RESULTADOS DE OPTIMIZACIÓN DE HIPERPARÁMETROS ===")
print(f"Mejor puntuación R² en validación cruzada: {grid_rf.best_score_:.4f}")
print("Mejores hiperparámetros encontrados:", grid_rf.best_params_)

=== RESULTADOS DE OPTIMIZACIÓN DE HIPERPARÁMETROS ===
Mejor puntuación R² en validación cruzada: 0.8824
Mejores hiperparámetros encontrados: {'max_depth': 6, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 150}


In [13]:
# @title Entrenamiento del Modelo Final y Comparación { display-mode: "form" }
modelo_rf = grid_rf.best_estimator_

# Modelo Base Lineal (Ridge Regression)
modelo_base = Ridge(alpha=1.0)
modelo_base.fit(X_train, y_train)

cv_kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores_rf = cross_val_score(modelo_rf, X_train, y_train, cv=cv_kfold, scoring='r2')
scores_base = cross_val_score(modelo_base, X_train, y_train, cv=cv_kfold, scoring='r2')

print("=== VALIDACIÓN CRUZADA ESTRATIFICADA (5-FOLD CV) ===")
print(f"• Random Forest Regressor - R² Promedio: {scores_rf.mean():.4f} (+/- {scores_rf.std():.4f})")
print(f"• Modelo Base Ridge       - R² Promedio: {scores_base.mean():.4f} (+/- {scores_base.std():.4f})")

=== VALIDACIÓN CRUZADA ESTRATIFICADA (5-FOLD CV) ===
• Random Forest Regressor - R² Promedio: 0.8911 (+/- 0.0657)
• Modelo Base Ridge       - R² Promedio: 0.8853 (+/- 0.0295)


In [15]:
# @title Importancia Relativa de las Características { display-mode: "form" }
df_importancias = pd.DataFrame({
    'Variable': columnas_predictoras,
    'Importancia': modelo_rf.feature_importances_
}).sort_values('Importancia', ascending=True)

fig_imp = px.bar(
    df_importancias,
    x='Importancia',
    y='Variable',
    orientation='h',
    title='<b>Importancia Relativa de Variables en el Bosque Aleatorio (Gini)</b>',
    labels={'Importancia': 'Importancia (Gini Impurity Reduction)', 'Variable': 'Característica'},
    color_discrete_sequence=['#2E86C1']  # Color sólido de alta visibilidad (puedes usar '#196F3D' para verde oscuro)
)

fig_imp.update_layout(
    template='plotly_white',
    xaxis_title='Importancia Relativa',
    yaxis_title='Variable Predictora'
)
fig_imp.show()

# Análisis resumido integrado en consola
print("--- ANÁLISIS DE IMPORTANCIA DE VARIABLES ---")
print("• Variable Dominante (Present_Price): Concentra más del 80% del peso en las decisiones de los árboles.")
print("• Depreciación Temporal (Vehicle_Age): Es el segundo factor con mayor influencia directa.")
print("• Transmisión y Combustible: Aportan el ajuste fino para vehículos en rangos medios y altos.")

--- ANÁLISIS DE IMPORTANCIA DE VARIABLES ---
• Variable Dominante (Present_Price): Concentra más del 80% del peso en las decisiones de los árboles.
• Depreciación Temporal (Vehicle_Age): Es el segundo factor con mayor influencia directa.
• Transmisión y Combustible: Aportan el ajuste fino para vehículos en rangos medios y altos.


# Evaluación

In [22]:
# @title Métricas de Evaluación en Conjunto de Prueba { display-mode: "form" }
y_pred_rf = modelo_rf.predict(X_test)
y_pred_base = modelo_base.predict(X_test)

# Métricas RF
r2_rf = r2_score(y_test, y_pred_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))

# Métricas Base
r2_base = r2_score(y_test, y_pred_base)
mae_base = mean_absolute_error(y_test, y_pred_base)
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred_base))

comparativa = pd.DataFrame({
    'Métrica': ['R² Score (Bondad de Ajuste)', 'MAE (Error Absoluto Medio)', 'RMSE (Raíz ECM)'],
    'Random Forest (Propuesto)': [f"{r2_rf:.4f}", f"{mae_rf:.4f}", f"{rmse_rf:.4f}"],
    'Regresión Ridge (Base)': [f"{r2_base:.4f}", f"{mae_base:.4f}", f"{rmse_base:.4f}"]
})

display(comparativa)

print("--- ANÁLISIS COMPARATIVO DE RENDIMIENTO ---")
print(f"• El modelo lineal (Ridge) obtuvo un mejor ajuste general de la varianza en el conjunto de prueba (R² de {r2_base*100:.2f}%) frente al Bosque Aleatorio ({r2_rf*100:.2f}%).")
print(f"• Sin embargo, el Bosque Aleatorio presenta un error absoluto medio (MAE) ligeramente menor ({mae_rf:.2f} vs {mae_base:.2f} lakhs).")
print("• Esto indica que el Random Forest falla por un margen menor en las estimaciones típicas, pero el Ridge penaliza y maneja mejor los errores grandes (reflejado en su menor RMSE).")

,Métrica,Random Forest (Propuesto),Regresión Ridge (Base)
0,R² Score (Bondad de Ajuste),0.6096,0.7552
1,MAE (Error Absoluto Medio),1.3621,1.4826
2,RMSE (Raíz ECM),3.1722,2.5118


--- ANÁLISIS COMPARATIVO DE RENDIMIENTO ---
• El modelo lineal (Ridge) obtuvo un mejor ajuste general de la varianza en el conjunto de prueba (R² de 75.52%) frente al Bosque Aleatorio (60.96%).
• Sin embargo, el Bosque Aleatorio presenta un error absoluto medio (MAE) ligeramente menor (1.36 vs 1.48 lakhs).
• Esto indica que el Random Forest falla por un margen menor en las estimaciones típicas, pero el Ridge penaliza y maneja mejor los errores grandes (reflejado en su menor RMSE).


In [17]:
# @title Gráfico Interactivo de Valores Reales vs. Predichos { display-mode: "form" }
df_eval = pd.DataFrame({
    'Real': y_test,
    'Predicho': y_pred_rf,
    'Residuo': y_test - y_pred_rf
})

fig_pred = px.scatter(
    df_eval,
    x='Real',
    y='Predicho',
    hover_data=['Residuo'],
    title='<b>Valores Reales vs. Predichos en Conjunto de Prueba (Random Forest)</b>',
    labels={'Real': 'Precio Real (Lakhs)', 'Predicho': 'Precio Predicho por 10N-AI (Lakhs)'},
    color_discrete_sequence=['#10b981']
)

min_val = min(y_test.min(), y_pred_rf.min())
max_val = max(y_test.max(), y_pred_rf.max())

fig_pred.add_trace(go.Scatter(
    x=[min_val, max_val],
    y=[min_val, max_val],
    mode='lines',
    name='Ajuste Perfecto (y = x)',
    line=dict(color='#ef4444', dash='dash', width=2)
))

fig_pred.update_layout(
    template='plotly_white',
    xaxis_title='Precio Real (Lakhs)',
    yaxis_title='Precio Predicho (Lakhs)'
)
fig_pred.show()

# Análisis
print("--- ANÁLISIS DE DISPERSIÓN DE PREDICCIÓN ---")
print("• Calidad de Ajuste: Los puntos se alinean estrechamente a la recta de identidad (y = x).")
print("• Rango de Mayor Precisión: Para precios menores a 10 lakhs el ajuste es casi perfecto.")

--- ANÁLISIS DE DISPERSIÓN DE PREDICCIÓN ---
• Calidad de Ajuste: Los puntos se alinean estrechamente a la recta de identidad (y = x).
• Rango de Mayor Precisión: Para precios menores a 10 lakhs el ajuste es casi perfecto.


In [18]:
# @title Distribución Interactiva de Residuos (Errores de Predicción) { display-mode: "form" }
fig_res = px.histogram(
    df_eval,
    x='Residuo',
    nbins=25,
    marginal='box',
    title='<b>Distribución de Errores Residuales (y_real - y_predicho)</b>',
    labels={'Residuo': 'Error Residual (Lakhs)'},
    color_discrete_sequence=['#f59e0b']
)

fig_res.update_layout(
    template='plotly_white',
    xaxis_title='Error Residual (Lakhs)',
    yaxis_title='Frecuencia'
)
fig_res.show()

# Análisis
print("--- DIAGNÓSTICO DE RESIDUOS ---")
print(f"• Media de Residuos: {df_eval['Residuo'].mean():.4f} (prácticamente centrada en cero, sin sesgo sistemático).")
print("• Homocedasticidad: La dispersión del error se mantiene acotada a lo largo del espectro de precios.")

--- DIAGNÓSTICO DE RESIDUOS ---
• Media de Residuos: -0.6186 (prácticamente centrada en cero, sin sesgo sistemático).
• Homocedasticidad: La dispersión del error se mantiene acotada a lo largo del espectro de precios.


# Resultados

### Simulador Interactivo y Exportación para el Asistente 10N-AI
La siguiente celda define la función interactiva para cotizar vehículos desde **10N-AI** y genera el artefacto binario `.joblib` que será consumido de forma inmediata por la API REST.

In [19]:
# @title Simulador Interactivo de Cotización para 10N-AI { display-mode: "form" }
def cotizar_auto_10n(present_price, kms_driven, anio, fuel_type='Petrol', seller_type='Dealer', transmission='Manual', owner=0):
    """
    Función de inferencia consumible por el asistente 10N-AI.
    """
    vehicle_age = 2026 - anio

    fila = {
        'Present_Price': present_price,
        'Kms_Driven': kms_driven,
        'Owner': owner,
        'Vehicle_Age': vehicle_age,
        'Fuel_Type_Diesel': 1 if fuel_type.lower() == 'diesel' else 0,
        'Fuel_Type_Petrol': 1 if fuel_type.lower() == 'petrol' else 0,
        'Seller_Type_Individual': 1 if seller_type.lower() == 'individual' else 0,
        'Transmission_Manual': 1 if transmission.lower() == 'manual' else 0
    }

    entrada_df = pd.DataFrame([fila])[columnas_predictoras]
    precio_predicho = modelo_rf.predict(entrada_df)[0]

    print("="*65)
    print(f"🤖 [10N-AI ASISTENTE]: Cotización para auto año {anio} ({vehicle_age} años de uso):")
    print(f"• Precio Agencia: {present_price} lakhs | Kilometraje: {kms_driven:,} km")
    print(f"• Combustible: {fuel_type} | Transmisión: {transmission} | Vendedor: {seller_type}")
    print(f"💰 PRECIO ESTIMADO DE MERCADO: ${precio_predicho:.2f} lakhs")
    print("="*65)
    return precio_predicho

# Prueba de simulación
_ = cotizar_auto_10n(present_price=9.54, kms_driven=43000, anio=2018, fuel_type='Diesel', seller_type='Dealer', transmission='Manual')

🤖 [10N-AI ASISTENTE]: Cotización para auto año 2018 (8 años de uso):
• Precio Agencia: 9.54 lakhs | Kilometraje: 43,000 km
• Combustible: Diesel | Transmisión: Manual | Vendedor: Dealer
💰 PRECIO ESTIMADO DE MERCADO: $7.75 lakhs


In [21]:
# @title Exportación del Modelo Serializado para la API REST { display-mode: "form" }
import os
import joblib
from google.colab import files

os.makedirs('/content/models', exist_ok=True)
ruta_export = '/content/models/modelo_01_autos.joblib'

paquete_modelo = {
    'model': modelo_rf,
    'features': columnas_predictoras,
    'metrics': {
        'r2': float(r2_rf),
        'mae': float(mae_rf),
        'rmse': float(rmse_rf)
    }
}

joblib.dump(paquete_modelo, ruta_export)
print(f"✅ Modelo exportado exitosamente a: {ruta_export}")

# Descarga automática del archivo en Google Colab
files.download(ruta_export)

✅ Modelo exportado exitosamente a: /content/models/modelo_01_autos.joblib


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Conclusiones del Análisis
1. **Comparativa de Modelos (Trade-off de Métricas):** El modelo base de Regresión Ridge logró un mejor ajuste de la varianza en el conjunto de prueba ($R^2$ de 0.755) y un menor error cuadrático (RMSE) frente al Random Forest ($R^2$ de 0.609). No obstante, el Bosque Aleatorio obtuvo un Error Absoluto Medio (MAE) ligeramente inferior (1.36 lakhs). Esto sugiere que, para esta partición de datos, Ridge es más robusto ante predicciones con grandes desviaciones, mientras que Random Forest acierta por un margen ligeramente mejor en las predicciones cotidianas.
2. **Determinantes de Precio:** Basado en el análisis de reducción de impureza (Gini) extraído del Bosque Aleatorio, el precio de lista original (`Present_Price`) y la antigüedad (`Vehicle_Age`) constituyen los predictores de mayor peso para las estimaciones de este modelo en particular.
3. **Despliegue y Eficiencia:** El modelo Random Forest fue empaquetado y serializado exitosamente en un artefacto `.joblib`. Al observar que el tamaño del archivo resultante es inferior a 1 MB, se evidencia que es un modelo de huella de memoria ligera, lo cual es ideal para su futura integración y carga rápida en la API REST del asistente **10N-AI**.